# Diabetic Retinopathy Preprocessing
Pipeline for retinal fundus preprocessing to reduce Dataset size.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

## Configuration

In [ ]:
BASE_DIR = Path("data")     # directory containing raw images
PREP_DIR = Path("dataset")  # directory to save preprocessed images and labels

(PREP_DIR / "train").mkdir(parents=True, exist_ok=True)
(PREP_DIR / "test").mkdir(parents=True, exist_ok=True)

IMG_SIZE = 300

## Fundus Preprocessing (Circular Crop + CLAHE)
Circular crop chosen because it was a popular suggestion by the DR Detection Kaggle competition winners. \
CLAHE chosen by searching "improve contrast problems in image processing" in google and gemini's response to it.\
Additional src: https://docs.opencv.org/4.x/d5/daf/tutorial_py_histogram_equalization.html

In [ ]:
def detect_retinal_circle(gray):
    _threshold, threshimg = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)         #https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html keeping threshold=10
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))                 #https://opencv24-python-tutorials.readthedocs.io/en/latest/py_tutorials/py_imgproc/py_morphological_ops/py_morphological_ops.html circular eye
    threshimg = cv2.morphologyEx(threshimg, cv2.MORPH_CLOSE, kernel)                #https://docs.opencv.org/4.x/d9/d61/tutorial_py_morphological_ops.html closing to fill small holes in the eye region
    threshimg = cv2.morphologyEx(threshimg, cv2.MORPH_OPEN, kernel)                 #https://docs.opencv.org/4.x/d9/d61/tutorial_py_morphological_ops.html opening to remove small noise outside the eye region

    contours, _ = cv2.findContours(threshimg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)   #https://docs.opencv.org/4.x/d4/d73/tutorial_py_contours_begin.html find contours of the eye region
    if not contours:
        return None

    (cx, cy), r = cv2.minEnclosingCircle(max(contours, key=cv2.contourArea))        #https://docs.opencv.org/3.4/dd/d49/tutorial_py_contour_features.html find the minimum enclosing circle of the largest contour
    return int(cx), int(cy), int(r)


def apply_clahe(bgr):
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))                      #https://docs.opencv.org/4.x/d5/daf/tutorial_py_histogram_equalization.html apply CLAHE
    lab = cv2.merge([clahe.apply(l), a, b])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def preprocess_image(path):
    bgr = cv2.imread(str(path))
    if bgr is None:
        return None

    h, w = bgr.shape[:2]
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    circle = detect_retinal_circle(gray)

    if circle is not None:
        cx, cy, r = circle
        half = int(r * 1.05)

        x1 = max(cx-half,0)
        x2 = min(cx+half,w)
        y1 = max(cy-half,0)
        y2 = min(cy+half,h)

        side = min(x2-x1, y2-y1)
        mx = (x1+x2)//2
        my = (y1+y2)//2

        x1 = max(mx-side//2,0)
        x2 = x1+side
        y1 = max(my-side//2,0)
        y2 = y1+side
    else:
        side = min(h,w)
        x1 = (w-side)//2; x2 = x1+side
        y1 = (h-side)//2; y2 = y1+side

    crop = bgr[y1:y2, x1:x2]
    enhanced = apply_clahe(crop)
    resized = cv2.resize(enhanced, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LANCZOS4)

    return cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)


## Batch Preprocess Images

In [ ]:
def preprocess_folder(input_dir, output_dir):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    images = list(input_dir.glob("*.jpeg")) + list(input_dir.glob("*.jpg")) + list(input_dir.glob("*.png"))

    for img_path in tqdm(images):
        out_path = output_dir / (img_path.stem + ".jpeg")

        rgb = preprocess_image(img_path)
        if rgb is None:
            continue

        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        cv2.imwrite(str(out_path), bgr)
